In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [15]:
# 1. Load the dataset
def load_housing_data():
    return pd.read_csv("../data/housing.csv")

housing = load_housing_data()

# FileNotFoundError fix:
# Current folder = src/, file is in data/
# Use "../data/housing.csv" (.. = go up one level)

In [16]:
# 2. Create a stratified test set to avoid sampling bias
# We use median_income to create an income category for stratification
housing['income_cat'] = pd.cut(housing["median_income"], 
                               bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf], 
                               labels=[1, 2, 3, 4, 5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing['income_cat']):
    strat_train_set = housing.loc[train_index].drop("income_cat", axis=1)
    strat_test_set = housing.loc[test_index].drop("income_cat", axis=1)

# Work on a copy of the training data
housing = strat_train_set.copy()

In [17]:
# 3. Separate features and labels
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

In [18]:
# 4. List the numerical and categorical columns
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]

In [23]:
# 5. Building the Preprocessing Pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs),
])

# Prepare the data
housing_prepared = full_pipeline.fit_transform(housing)

In [24]:
# 6. Model Training and Evaluation using Cross-Validation
def display_scores(scores, model_name):
    print(f"\n--- {model_name} ---")
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

# Linear Regression
lin_reg = LinearRegression()
lin_scores = -cross_val_score(lin_reg, housing_prepared, housing_labels, 
                             scoring="neg_root_mean_squared_error", cv=10)
display_scores(lin_scores, "Linear Regression")

# Decision Tree Regressor
tree_reg = DecisionTreeRegressor(random_state=42)
tree_scores = -cross_val_score(tree_reg, housing_prepared, housing_labels, 
                             scoring="neg_root_mean_squared_error", cv=10)
display_scores(tree_scores, "Decision Tree")

# Random Forest Regressor
forest_reg = RandomForestRegressor(random_state=42)
forest_scores = -cross_val_score(forest_reg, housing_prepared, housing_labels, 
                                scoring="neg_root_mean_squared_error", cv=10)
display_scores(forest_scores, "Random Forest")


--- Linear Regression ---
Scores: [72229.03469752 65318.2240289  67706.39604745 69368.53738998
 66767.61061621 73003.75273869 70522.24414582 69440.77896541
 66930.32945876 70756.31946074]
Mean: 69204.32275494766
Standard deviation: 2372.0707910559195

--- Decision Tree ---
Scores: [71177.6601991  69770.07865373 64770.5639395  68536.60203993
 67057.08155801 68847.12456973 70977.38255647 69208.86346929
 67187.87131535 73280.38732407]
Mean: 69081.361562518
Standard deviation: 2296.288087393378

--- Random Forest ---
Scores: [51039.08053738 48741.94041426 45940.42771745 50501.41453432
 47387.7896427  49595.25845731 51625.68567717 48865.70709952
 47322.87631489 53301.08748462]
Mean: 49432.12678796127
Standard deviation: 2124.8587921578355


In [25]:
# 7. Final Training on the full training set with the best model
forest_reg.fit(housing_prepared, housing_labels)
print("\nFinal Model (Random Forest) trained successfully.")


Final Model (Random Forest) trained successfully.
